# LANGFENS — RQ1-TRN-03B: Arm 2A Focal BCE Smoke Execution on Kaggle Tesla T4x2
Official 1-epoch smoke execution notebook for Arm 2A Focal BCE Pilot (`KAGGLE_T4_CUDA_FP16_AMP`).

## Operational Role & Safety Invariants
- **Role:** Standalone 1-epoch execution smoke test (strictly capped at 1 completed epoch / ~600 updates).
- **Execution Cap:** Strictly 1 epoch via `--execution-cap-epochs 1`.
- **Run ID Isolation:** Uses dedicated run ID `pilot_arm2a_focal_smoke_1epoch`.
- **Mandatory Preflight First:** Preflight verification must pass before training starts.
- **Hardware Target:** Tesla T4 × 2 GPU (`Driver: >=580`, `CUDA: >=13.0`, 15,360 MiB VRAM per GPU).
- **Dataset Staging:** Authorized Fold 1 Train+Val package (`minkhoaa/langfens-rq1-fold1-staging`).
- **Sealed Test Isolation:** Active test speakers (`ABA`, `ASI`, `EBVS`, `HJK`, `LXC`, `PNV`) are 100% physically absent.
- **Manifest Integrity:** Verifies split manifest SHA-256 (`7d6bfdf62bc6702b449048968f20e8c78df5b174be341b8d081667982980804b`).
- **Dataset Integrity:** Verifies fold package SHA-256 (`b750eb3bf3971afccd9ab9058a64fa6b3f5781399b9ef0f1294d29bf580784a9`).
- **Remote Persistence Mirror:** Mirrors per-epoch checkpoints to private Hugging Face repo `minkhoaa/langfens-rq1-cross-fold-checkpoints` with SHA-256 readback.
- **Reviewed Base Commit:** `051a87071279a1f2662fa9c6c449339281e05063`.


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Checkout Reviewed Training Commit & Verify Tree
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

PINNED_COMMIT_SHA = "051a87071279a1f2662fa9c6c449339281e05063"
print(f"[GIT] Checking out pinned training commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip().lower()
print(f"[GIT] Verified Training Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA.lower(), (
    f"FAIL-CLOSED: Commit SHA mismatch: actual {actual_head} != expected {PINNED_COMMIT_SHA}"
)

status = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip()
assert len(status) == 0, f"Dirty tree: {status}"
print("[GIT] Repository verified clean at pinned training commit.")

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))


In [ ]:
# Phase 4: Mandatory Preflight Check (Fail-Closed Gate)
import subprocess, sys
from pathlib import Path

staging_input = Path("/kaggle/input/langfens-rq1-fold1-staging")
print(f"[PREFLIGHT-GATE] Auditing dataset input directory: {staging_input}")
if staging_input.is_dir():
    print(f"[PREFLIGHT-GATE] Directory contents: {[p.name for p in staging_input.iterdir()]}")
else:
    print(f"[PREFLIGHT-GATE WARNING] {staging_input} does not exist yet (requires dataset attachment).")

preflight_cmd = [
    sys.executable,
    "deploy/kaggle/kaggle_entrypoint.py",
    "--mode", "verify_pilot_arm2a",
    "--commit-sha", PINNED_COMMIT_SHA,
    "--kaggle-input-dir", str(staging_input),
    "--remote-mirror-backend", "huggingface",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
]
print("[PREFLIGHT-GATE] Executing mandatory preflight verification:")
print("  " + " ".join(preflight_cmd))
subprocess.run(preflight_cmd, check=True)
print("=" * 80)
print("[PREFLIGHT] Preflight verification PASSED successfully. Proceeding to smoke training...")
print("=" * 80)


In [ ]:
# Phase 5: Isolated Smoke Training Execution (1 epoch, run ID pilot_arm2a_focal_smoke_1epoch)
import subprocess, sys
from pathlib import Path

print("=" * 80)
print("[SMOKE LAUNCH] Starting Arm 2A Focal Smoke Training on Tesla T4...")
print(f"[SMOKE LAUNCH] Run ID: pilot_arm2a_focal_smoke_1epoch")
print(f"[SMOKE LAUNCH] Execution Cap: 1 completed epoch (~600 updates)")
print(f"[SMOKE LAUNCH] Pinned Commit SHA: {PINNED_COMMIT_SHA}")
print("=" * 80)

smoke_cmd = [
    sys.executable,
    "deploy/kaggle/kaggle_entrypoint.py",
    "--mode", "train_pilot_arm2a",
    "--commit-sha", PINNED_COMMIT_SHA,
    "--kaggle-input-dir", str(staging_input),
    "--run-id", "pilot_arm2a_focal_smoke_1epoch",
    "--execution-cap-epochs", "1",
    "--remote-mirror-backend", "huggingface",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
]
print("[SMOKE LAUNCH] Executing checked command:")
print("  " + " ".join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)
print("[SMOKE LAUNCH] Arm 2A Focal Smoke training subprocess completed successfully.")


In [ ]:
# Phase 6: Post-Smoke Artifact & Metric Verification
import json
from pathlib import Path

smoke_dir = Path("/kaggle/working/checkpoints/RQ1-TRN-03B/pilot_arm2a_focal_smoke_1epoch")
print(f"[VERIFY SMOKE] Checking smoke directory: {smoke_dir}")
assert smoke_dir.is_dir(), f"Smoke output directory missing: {smoke_dir}"
assert (smoke_dir / "last.pt").is_file(), f"Missing last.pt in {smoke_dir}"
assert (smoke_dir / "best.pt").is_file(), f"Missing best.pt in {smoke_dir}"
assert (smoke_dir / "history.json").is_file(), f"Missing history.json in {smoke_dir}"

hist = json.loads((smoke_dir / "history.json").read_text())
assert len(hist) == 1, f"Expected exactly 1 completed epoch, got {len(hist)}"
ep1 = hist[0]
print("=" * 80)
print("[SMOKE EXECUTION COMPLETED SUCCESSFULLY]")
print(f"Epoch: {ep1.get('epoch')}")
print(f"Updates: {ep1.get('successful_updates')}")
print(f"Train Binary Loss: {ep1.get('train_binary_loss')}")
print(f"Val AP: {ep1.get('val_ap')}")
print(f"Val CTC PER: {ep1.get('val_ctc_per')}")
print("=" * 80)
